# Module 3.19 — Mesh Generation

**The central engineering problem:** Your solver is only as good as your mesh. A beautiful turbulence model on a bad mesh gives wrong answers. A simple first-order scheme on a good mesh beats a high-order scheme on a bad one.

**Roadmap:**
1. Structured vs unstructured meshes — when to use which
2. Mesh quality metrics — aspect ratio, skewness, orthogonality, expansion ratio
3. y+ estimation — how to set first-cell height before running
4. Clustered grids in Python — geometric and tanh stretching
5. Quality checking — compute and visualise your mesh quality

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import PolyCollection

## 1. Structured vs Unstructured Meshes

### Structured mesh

Grid lines form a regular pattern — every interior node has the same number of neighbours (4 in 2D, 6 in 3D). Nodes are addressed by index (i, j) like a 2D array.

```
j=2  *---*---*---*
     |   |   |   |
j=1  *---*---*---*
     |   |   |   |
j=0  *---*---*---*
     i=0 i=1 i=2
```

**Advantages:**
- Simple indexing — `u[i,j]` accesses neighbour `u[i+1,j]` trivially
- Cache-friendly — memory layout matches spatial layout
- Natural for body-fitted grids (O-grid around cylinder, C-grid around airfoil)
- Solvers (TDMA, multigrid) exploit the regular structure

**Disadvantages:**
- Hard to handle complex geometry — you must parametrise the domain
- Topology must match geometry — an O-grid wastes cells far from the body

---

### Unstructured mesh

Cells can be any shape (triangles, quads in 2D; tetrahedra, hexahedra in 3D). Connectivity is stored explicitly in a list, not implied by (i,j) index.

**Advantages:**
- Handles ANY geometry automatically — mesh generators (GMSH, Pointwise) fill complex domains
- Local refinement is trivial — add cells only where needed
- Industry standard for external aerodynamics, biomedical flows, turbomachinery

**Disadvantages:**
- Connectivity lookup is expensive (cache misses)
- Higher-order schemes are harder to implement
- Mesh generation itself requires expertise

---

### Decision guide

| Flow | Mesh type | Reason |
|------|-----------|--------|
| Channel, cavity, pipe | Structured | Simple geometry, exploit regularity |
| Cylinder, sphere | Structured O-grid | Body-fitted, smooth near-wall layers |
| Airfoil | Structured C-grid | Resolves leading edge + wake |
| Engine, turbine blade | Unstructured | Complex geometry, automated meshing |
| Human artery | Unstructured | Impossible to parametrise by hand |

## 2. Mesh Quality Metrics

A mesh cell is "good" if it is close to equilateral, aligned with the flow, and smoothly transitions to its neighbours. Four key metrics:

---

### 2a. Aspect Ratio

$$AR = \frac{\text{longest edge}}{\text{shortest edge}}$$

- AR = 1: perfect square/equilateral — best accuracy
- AR >> 1: highly stretched cell — acceptable **only** when the flow gradients are aligned with the stretch direction

**Example:** Near a flat plate at Re = 10^6, the boundary layer is 1 mm thick but the plate is 1 m long. You NEED AR ~ 1000 in the wall-normal direction — and that is fine because the velocity only varies in the wall-normal direction.

**Rule:** High AR is acceptable when the long axis is aligned with the **flow direction** (gradients are small there). High AR across the flow (wall-normal) is dangerous.

---

### 2b. Skewness

How far a cell deviates from equilateral. Defined as:

```
skewness = (theta_max - theta_ideal) / (180 - theta_ideal)
```

where theta_ideal = 60 deg (triangle) or 90 deg (quad).

- Skewness = 0: perfect cell
- Skewness > 0.85: unacceptable — solver may diverge
- Skewness > 0.95: guaranteed convergence failure in most solvers

Skewness causes the face normal to point away from the line joining cell centres — the diffusion gradient is computed incorrectly.

---

### 2c. Orthogonality

The angle between the face normal and the line joining adjacent cell centres.

- 90 deg: perfect (face normal IS the cell-centre line)
- < 70 deg: start to worry
- < 45 deg: needs correction terms (non-orthogonal correction in OpenFOAM)

Low orthogonality near curved walls is common — mitigated by adding correction fluxes or by using better smoothing algorithms.

---

### 2d. Expansion Ratio (Growth Rate)

The ratio of adjacent cell sizes:

```
r = dy_{i+1} / dy_i
```

- r = 1: uniform (no stretching)
- r = 1.05 to 1.2: good practice — gradual transition
- r > 1.5: too sudden — solution jumps across the interface, introducing error
- Typical target: r < 1.2 for production simulations

**The rule of thumb:** never let consecutive cells differ in size by more than 20%.

In [ ]:
# ── Visualise the 4 quality metrics with simple examples ─────────

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# ── 1. Aspect ratio ────────────────────────────────────────────────
ax = axes[0, 0]
# Good cell (AR=1) and bad cell (AR=10)
good = plt.Polygon([[0,0],[1,0],[1,1],[0,1]], fill=True, facecolor='lightgreen', edgecolor='k', lw=2)
bad  = plt.Polygon([[2,0],[12,0],[12,1],[2,1]], fill=True, facecolor='salmon', edgecolor='k', lw=2)
ax.add_patch(good); ax.add_patch(bad)
ax.text(0.5, 0.5, 'AR = 1', ha='center', va='center', fontsize=11, fontweight='bold')
ax.text(7.0, 0.5, 'AR = 10', ha='center', va='center', fontsize=11, fontweight='bold')
ax.set_xlim(-0.5, 13); ax.set_ylim(-0.5, 2)
ax.set_aspect('equal'); ax.set_title('Aspect Ratio', fontsize=13)
ax.axis('off')

# ── 2. Skewness ────────────────────────────────────────────────────
ax = axes[0, 1]
good_q = plt.Polygon([[0,0],[1,0],[1,1],[0,1]],     fill=True, facecolor='lightgreen', edgecolor='k', lw=2)
bad_q  = plt.Polygon([[2,0],[3.8,0],[3.5,1],[2.2,1]], fill=True, facecolor='salmon',    edgecolor='k', lw=2)
ax.add_patch(good_q); ax.add_patch(bad_q)
ax.text(0.5, 0.5, 'Skew = 0',    ha='center', va='center', fontsize=11, fontweight='bold')
ax.text(2.9, 0.5, 'High skew',   ha='center', va='center', fontsize=11, fontweight='bold')
ax.set_xlim(-0.5, 4.5); ax.set_ylim(-0.5, 2)
ax.set_aspect('equal'); ax.set_title('Skewness', fontsize=13)
ax.axis('off')

# ── 3. Orthogonality ───────────────────────────────────────────────
ax = axes[1, 0]
# Two cells sharing a face; draw face normal vs cell-centre line
ax.fill([0,1,1,0], [0,0,1,1], color='lightgreen', alpha=0.5, zorder=1)
ax.fill([1,2,2,1], [0,0,1,1], color='lightgreen', alpha=0.5, zorder=1)
ax.plot([0,1,2],[0,0,0],'k-',lw=2); ax.plot([0,1,2],[1,1,1],'k-',lw=2)
ax.plot([0,0],[0,1],'k-',lw=2); ax.plot([1,1],[0,1],'k-',lw=2); ax.plot([2,2],[0,1],'k-',lw=2)
ax.annotate('', xy=(1,0.5), xytext=(0.5,0.5), arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.annotate('', xy=(1,0.5), xytext=(1.5,0.5), arrowprops=dict(arrowstyle='->', color='red',  lw=2))
ax.text(0.5, 0.75, 'cell A', ha='center', fontsize=10)
ax.text(1.5, 0.75, 'cell B', ha='center', fontsize=10)
ax.text(0.5, 0.3, 'd_PA', ha='center', fontsize=9, color='blue')
ax.text(1.5, 0.3, 'd_PB', ha='center', fontsize=9, color='red')
ax.text(1.0, -0.25, 'Face normal perpendicular to d_PA + d_PB = 90 deg (perfect)', ha='center', fontsize=9)

ax.fill([3,4,4.5,3.5], [0,0,1,1], color='salmon', alpha=0.5, zorder=1)
ax.fill([4,5,5.5,4.5], [0,0,1,1], color='salmon', alpha=0.5, zorder=1)
ax.plot([3,4,5,5.5],[0,0,0,0],'k-',lw=2)
ax.plot([3.5,4.5,5.5],[1,1,1],'k-',lw=2)
ax.plot([3,3.5],[0,1],'k-',lw=2); ax.plot([4,4.5],[0,1],'k-',lw=2); ax.plot([5,5.5],[0,1],'k-',lw=2)
ax.text(4.25, -0.25, 'Skewed face: normal not aligned with cell-centre line', ha='center', fontsize=9, color='red')

ax.set_xlim(-0.3, 6); ax.set_ylim(-0.6, 1.5)
ax.set_title('Orthogonality', fontsize=13); ax.axis('off')

# ── 4. Expansion ratio ─────────────────────────────────────────────
ax = axes[1, 1]
y_good = np.array([0, 0.05, 0.11, 0.18, 0.27, 0.39, 0.54, 0.72, 0.90, 1.0])
y_bad  = np.array([0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.40, 0.70, 0.85, 1.0])
for yi in y_good:
    ax.axhline(yi, color='green', lw=1.5, alpha=0.7)
for yi in y_bad:
    ax.axhline(yi + 1.3, color='red', lw=1.5, alpha=0.7)
ax.text(0.5, 0.5,  'Good: r ~ 1.15\n(smooth)', ha='center', va='center', fontsize=10, color='darkgreen')
ax.text(0.5, 1.85, 'Bad: sudden jump\n(r ~ 4 at midpoint)', ha='center', va='center', fontsize=10, color='darkred')
ax.set_xlim(0,1); ax.set_ylim(-0.1, 2.5); ax.set_title('Expansion Ratio', fontsize=13); ax.axis('off')

plt.suptitle('Four Key Mesh Quality Metrics', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. y+ Estimation — Before You Run

**The problem:** You need to know the first-cell height `y_1` to resolve the sublayer (y+ < 5 for low-Re model, y+ = 30–100 for wall functions). But `y+` depends on `u_tau`, which you don't know until you've run the simulation.

**The solution:** Estimate `u_tau` from an empirical skin-friction correlation.

---

### Flat plate (external aerodynamics)

For a turbulent flat plate at Reynolds number `Re_L = U·L/nu`:

```
C_f  ≈  0.027 · Re_L^(-1/7)        (Prandtl power law)
tau_w = C_f · (1/2) · rho · U^2
u_tau = sqrt(tau_w / rho)
y_1   = y+_target · nu / u_tau
```

### Pipe / channel (internal flow)

For fully-developed turbulent pipe flow:

```
C_f  ≈  0.079 · Re_D^(-1/4)        (Blasius correlation, Re_D < 100,000)
```

---

### The workflow

```
Given:  U (velocity), L or D (length scale), nu (viscosity), rho (density)
Target: y+_target (e.g. 1 for low-Re model, 50 for wall functions)

Step 1: Re  = U * L / nu
Step 2: C_f = 0.027 * Re^(-1/7)          (flat plate) or 0.079 * Re^(-0.25) (pipe)
Step 3: tau_w = 0.5 * rho * U^2 * C_f
Step 4: u_tau = sqrt(tau_w / rho)
Step 5: y_1   = y+_target * nu / u_tau
Step 6: Re_tau = u_tau * L_perp / nu      (check: matches y+_max)
```

In [ ]:
def estimate_y1(U, L, nu, rho=1.0, yplus_target=1.0, geometry='flat_plate'):
    """
    Estimate first-cell height y_1 for a target y+ value.
    geometry: 'flat_plate' or 'pipe'
    """
    Re = U * L / nu

    if geometry == 'flat_plate':
        Cf = 0.027 * Re**(-1/7)
    else:  # pipe / channel (Blasius)
        Cf = 0.079 * Re**(-0.25)

    tau_w  = 0.5 * rho * U**2 * Cf
    u_tau  = np.sqrt(tau_w / rho)
    y1     = yplus_target * nu / u_tau
    Re_tau = u_tau * L / nu

    return dict(Re=Re, Cf=Cf, tau_w=tau_w, u_tau=u_tau, y1=y1, Re_tau=Re_tau)

# ── Example 1: Aircraft wing at cruise ───────────────────────────
r = estimate_y1(U=250, L=5.0, nu=1.5e-5, rho=0.4, yplus_target=1.0)
print('=== Aircraft wing at cruise (y+ target = 1) ===')
print(f'  Re       = {r["Re"]:.2e}')
print(f'  Cf       = {r["Cf"]:.4f}')
print(f'  tau_w    = {r["tau_w"]:.2f} Pa')
print(f'  u_tau    = {r["u_tau"]:.3f} m/s')
print(f'  y_1      = {r["y1"]*1e6:.2f} micrometres  ← first cell height')
print(f'  Re_tau   = {r["Re_tau"]:.0f}')

print()

# ── Example 2: Water pipe at industrial flow rate ─────────────────
r2 = estimate_y1(U=3.0, L=0.1, nu=1e-6, rho=1000, yplus_target=50, geometry='pipe')
print('=== Water pipe D=0.1m, U=3m/s (wall functions, y+ target = 50) ===')
print(f'  Re       = {r2["Re"]:.2e}')
print(f'  Cf       = {r2["Cf"]:.4f}')
print(f'  tau_w    = {r2["tau_w"]:.2f} Pa')
print(f'  u_tau    = {r2["u_tau"]:.3f} m/s')
print(f'  y_1      = {r2["y1"]*1e3:.3f} mm  ← first cell height')
print(f'  Re_tau   = {r2["Re_tau"]:.0f}')

print()

# ── Sensitivity: how y_1 varies with Re ───────────────────────────
Re_vals = np.logspace(4, 8, 100)
nu = 1.5e-5; U = 1.0; L = 1.0; rho = 1.0
y1_vals = [estimate_y1(U, L*Re*nu/U, nu, rho, 1.0)['y1'] for Re in Re_vals]

plt.figure(figsize=(8, 4))
plt.loglog(Re_vals, np.array(y1_vals)*1000, 'b-', lw=2)
plt.xlabel('Re_L'); plt.ylabel('y_1  (mm)  for y+ = 1')
plt.title('Required first-cell height vs Reynolds number  (flat plate, y+ = 1)')
plt.grid(True, which='both', alpha=0.4)
plt.tight_layout(); plt.show()
print('Key insight: as Re doubles, y_1 roughly halves — high-Re meshes are VERY fine near walls')

## 4. Clustered Grids in Python

A uniform grid wastes points in the core. Two standard clustering strategies:

### Geometric (exponential) stretching

Each cell is `r` times larger than the previous:

```
dy_0  = first cell height
dy_i  = dy_0 * stretch^i
```

Sum must equal H: `dy_0 * (1 - stretch^N) / (1 - stretch) = H`

Solve for `dy_0` given `stretch` and `N`.

**Pros:** Simple, explicit formula for `dy_0`, very fine near wall
**Cons:** Growth rate not uniform — gets coarser faster than tanh

---

### Tanh stretching (standard for channel flow)

```
y_i = H * (1 - tanh(s * (1 - xi_i)) / tanh(s))
where xi_i = i / (N-1)   (uniform 0→1)
```

- `s = 0`: uniform grid
- `s = 2`: moderate clustering
- `s = 4`: strong clustering near y=0

**Pros:** Smooth, controllable, symmetric option available
**Cons:** Need to tune `s` to hit target `y+`

In [ ]:
def geometric_grid(N, H, stretch):
    """Geometric stretching: dy_{i+1} = stretch * dy_i."""
    dy0 = H * (1 - stretch) / (1 - stretch**(N-1))
    y = np.zeros(N)
    for i in range(1, N):
        y[i] = y[i-1] + dy0 * stretch**(i-1)
    return np.clip(y, 0, H)

def tanh_grid(N, H, s):
    """Tanh stretching: clustered near y=0, coarser near y=H."""
    xi = np.linspace(0, 1, N)
    return H * (1 - np.tanh(s * (1 - xi)) / np.tanh(s))

# ── Compare three grids ───────────────────────────────────────────
N = 50; H = 1.0

y_uniform  = np.linspace(0, H, N)
y_geom     = geometric_grid(N, H, stretch=1.08)
y_tanh     = tanh_grid(N, H, s=3.0)

# Expansion ratios
def expansion_ratio(y):
    dy = np.diff(y)
    return dy[1:] / dy[:-1]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

grids  = [y_uniform, y_geom,  y_tanh]
labels = ['Uniform', 'Geometric (r=1.08)', 'Tanh (s=3)']
colors = ['gray',    'steelblue',           'darkorange']

for ax, y, label, color in zip(axes, grids, labels, colors):
    # Draw horizontal grid lines
    for yi in y:
        ax.axhline(yi, color=color, lw=1.0, alpha=0.8)
    er = expansion_ratio(y)
    ax.set_title(f'{label}\nFirst cell: y_1={y[1]*1000:.2f} mm\nMax expansion ratio: {er.max():.3f}',
                 fontsize=10)
    ax.set_xlim(0, 1); ax.set_ylim(0, H)
    ax.set_xlabel('x (dummy)'); ax.set_ylabel('y (m)')
    ax.set_yticks(np.linspace(0, H, 6))

plt.suptitle(f'Grid comparison — N={N} points, H={H} m', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Print first few cell sizes ─────────────────────────────────────
print(f'{"Point":>6}  {"Uniform dy":>12}  {"Geometric dy":>14}  {"Tanh dy":>10}')
for i in range(min(8, N-1)):
    print(f'{i+1:>6}  {np.diff(y_uniform)[i]*1000:>10.3f} mm  {np.diff(y_geom)[i]*1000:>12.4f} mm  {np.diff(y_tanh)[i]*1000:>8.4f} mm')

## 5. Exercise — Design a Mesh for the Turbulent Channel

Use what you have learned to design a proper mesh for the turbulent channel from Module 3.18.

**Given:**
- Channel half-height: H = 0.05 m
- Bulk velocity: U = 10 m/s
- Kinematic viscosity: nu = 1.5e-5 m/s2 (air)
- Target: y+ = 1 at first cell (resolve the viscous sublayer)

**Tasks:**
1. Use `estimate_y1` to find the required first-cell height `y_1`
2. Use `geometric_grid` with a growth rate `r = 1.15` and N = 60 to build the grid
3. Plot the grid and check: does `y[1]` match your `y_1` estimate? If not, adjust `r` or `N`
4. Compute and print the expansion ratio — is it below 1.2 everywhere?

**Predict first:** Will you need more or fewer cells than the uniform N=500 from Module 3.18?